In [2]:
# Celda 1: Importaciones y configuración
import sys
import pandas as pd
import numpy as np
import re
from pathlib import Path

sys.path.append(str(Path('..').resolve()))
from src.config import SILVER_PATH, GOLD_PATH, get_silver_file, listar_tablas_silver, get_gold_file

# Diccionario para almacenar solo los DataFrames transformados (dim_* y fact_*)
dataframes = {}

print("Configuración completada. Silver path:", SILVER_PATH)
print("Tablas disponibles en silver:", listar_tablas_silver())

Configuración completada. Silver path: D:\proyectos de github\proyecto northwind\data\02_silver
Tablas disponibles en silver: ['customers', 'employees', 'employee_privileges', 'inventory_transactions', 'inventory_transaction_types', 'orders', 'orders_status', 'orders_tax_status', 'order_details', 'order_details_status', 'privileges', 'products', 'purchase_orders', 'purchase_order_details', 'sales_reports', 'shippers', 'strings', 'suppliers']


## Cargamos todos los archivos de la data/02_silver
_maneja errores en caso no encuentre una archivo_

In [3]:
# Celda 2: Carga de datos desde silver (versión con verificación previa)
print("Cargando datos desde silver...")

# Verificar qué tablas están disponibles en silver
tablas_silver = listar_tablas_silver()
print(f"Tablas encontradas en silver: {tablas_silver}")

# Diccionario que mapea nombres de archivo a nombres de variable
archivos_a_variables = {
    'customers': 'customers_df',
    'employees': 'employees_df',
    'employee_privileges': 'employee_privileges_df',
    'privileges': 'privileges_df',
    'products': 'products_df',
    'suppliers': 'suppliers_df',
    'orders': 'orders_df',
    'order_details': 'order_details_df',
    'shippers': 'shippers_df',
    'inventory_transactions': 'inventory_transactions_df',
    'inventory_transaction_types': 'inventory_transaction_types_df',
    'purchase_orders': 'purchase_orders_df',
    'purchase_order_details': 'purchase_order_details_df'
}

# Verificar que todos los archivos existen
archivos_faltantes = [archivo for archivo in archivos_a_variables.keys() if archivo not in tablas_silver]

if archivos_faltantes:
    print("❌ ERROR CRÍTICO: No se pueden cargar los siguientes archivos desde silver:")
    for archivo in archivos_faltantes:
        print(f"  - {archivo}.csv")
    print("\n🔍 Posibles soluciones:")
    print("  1. Ejecuta primero el proceso de limpieza (bronze -> silver)")
    print("  2. Verifica que los archivos existan en la carpeta 02_silver")
    print("  3. Revisa los nombres de los archivos (deben coincidir exactamente)")
    raise FileNotFoundError(f"No se encontraron {len(archivos_faltantes)} archivos necesarios en silver")
else:
    print("✅ Verificación completada: todos los archivos existen en silver.")

# Cargar todos los archivos
silver_data = {}
for archivo, var_name in archivos_a_variables.items():
    try:
        ruta_archivo = get_silver_file(archivo)
        df = pd.read_csv(ruta_archivo)
        silver_data[archivo] = df
        # Asignar a variable local dinámicamente
        globals()[var_name] = df
        print(f"✅ Cargado: {archivo}.csv -> {var_name} ({len(df)} registros)")
    except Exception as e:
        print(f"❌ Error cargando {archivo}.csv: {e}")
        raise

print("\n" + "="*60)
print("✅ TODOS LOS ARCHIVOS FUERON CARGADOS EXITOSAMENTE DESDE SILVER")
print("="*60)
print("\nVariables locales creadas:")
for var_name in archivos_a_variables.values():
    df = globals().get(var_name)
    if df is not None:
        print(f"  • {var_name}: {len(df)} registros")

Cargando datos desde silver...
Tablas encontradas en silver: ['customers', 'employees', 'employee_privileges', 'inventory_transactions', 'inventory_transaction_types', 'orders', 'orders_status', 'orders_tax_status', 'order_details', 'order_details_status', 'privileges', 'products', 'purchase_orders', 'purchase_order_details', 'sales_reports', 'shippers', 'strings', 'suppliers']
✅ Verificación completada: todos los archivos existen en silver.
✅ Cargado: customers.csv -> customers_df (29 registros)
✅ Cargado: employees.csv -> employees_df (9 registros)
✅ Cargado: employee_privileges.csv -> employee_privileges_df (1 registros)
✅ Cargado: privileges.csv -> privileges_df (1 registros)
✅ Cargado: products.csv -> products_df (45 registros)
✅ Cargado: suppliers.csv -> suppliers_df (10 registros)
✅ Cargado: orders.csv -> orders_df (48 registros)
✅ Cargado: order_details.csv -> order_details_df (58 registros)
✅ Cargado: shippers.csv -> shippers_df (3 registros)
✅ Cargado: inventory_transactions.

## Creación de Tablas Dimensionales y de Hechos

### Tabla dim_date

In [4]:
# Celda 3: Creación de dim_date - Dimensión de tiempo
print("Creando dim_date...")

# Extraer todas las fechas relevantes de los datasets
all_dates = pd.Series(dtype='datetime64[ns]')

# Fechas de orders
if orders_df is not None and 'order_date' in orders_df.columns:
    orders_dates = pd.to_datetime(orders_df['order_date'].dropna(), errors='coerce')
    all_dates = pd.concat([all_dates, orders_dates])

# Fechas de inventory_transactions
if inventory_transactions_df is not None and 'transaction_created_date' in inventory_transactions_df.columns:
    inv_dates = pd.to_datetime(inventory_transactions_df['transaction_created_date'].dropna(), errors='coerce')
    all_dates = pd.concat([all_dates, inv_dates])

# Fechas de purchase_orders
if purchase_orders_df is not None and 'creation_date' in purchase_orders_df.columns:
    po_dates = pd.to_datetime(purchase_orders_df['creation_date'].dropna(), errors='coerce')
    all_dates = pd.concat([all_dates, po_dates])

# Eliminar duplicados y NaT
all_dates = all_dates.dropna().drop_duplicates().sort_values().reset_index(drop=True)

# Crear dataframe de fechas
dim_date = pd.DataFrame({
    'sk_date': all_dates.dt.strftime('%Y%m%d').astype(int),
    'date_actual': all_dates,
    'year': all_dates.dt.year,
    'quarter': all_dates.dt.quarter,
    'month': all_dates.dt.month,
    'month_name': all_dates.dt.month_name(),
    'day': all_dates.dt.day,
    'day_name': all_dates.dt.day_name(),
    'week': all_dates.dt.isocalendar().week,
    'is_weekend': (all_dates.dt.dayofweek >= 5).astype(int)
})

# Guardar SOLO en el diccionario principal
dataframes['dim_date'] = dim_date

print(f"✅ dim_date creada: {len(dim_date)} registros")
print(f"Rango de fechas: {dim_date['date_actual'].min()} a {dim_date['date_actual'].max()}")

Creando dim_date...
✅ dim_date creada: 137 registros
Rango de fechas: 2006-01-15 00:00:00 a 2006-06-23 00:00:00


### Tabla dim_products

In [5]:
# Celda 4: Creación de dim_products - Productos (aplanada con categorías y proveedores)
print("Creando dim_products...")

if products_df is not None and suppliers_df is not None:
    # Función para parsear supplier_ids (pueden ser listas como "[2, 6]")
    def parse_supplier_ids(supplier_ids_str):
        if pd.isna(supplier_ids_str) or supplier_ids_str == '':
            return []
        # Extraer números de la cadena
        numbers = re.findall(r'\d+', str(supplier_ids_str))
        return [int(num) for num in numbers]
    
    products_df = products_df.copy()
    products_df['supplier_ids_list'] = products_df['supplier_ids'].apply(parse_supplier_ids)
    
    # Para simplificar, tomamos el primer proveedor si existe
    products_df['primary_supplier_id'] = products_df['supplier_ids_list'].apply(
        lambda x: x[0] if len(x) > 0 else None
    )
    
    # Merge con proveedores para obtener nombre de empresa
    products_with_supplier = products_df.merge(
        suppliers_df[['id', 'company']], 
        left_on='primary_supplier_id', 
        right_on='id', 
        how='left',
        suffixes=('', '_supplier')
    )
    
    dim_products = pd.DataFrame({
        'sk_product': range(1, len(products_df) + 1),  # Clave subrogada
        'product_id': products_df['id'],
        'product_name': products_df['product_name'],
        'category': products_df['category'],
        'supplier_company': products_with_supplier['company'],  # Denormalizado
        'list_price': products_df['list_price'],
        'standard_cost': products_df['standard_cost'],
        'quantity_per_unit': products_df['quantity_per_unit'],
        'discontinued': products_df['discontinued']
    })
    
    # Guardar SOLO en el diccionario principal
    dataframes['dim_products'] = dim_products
    
    print(f"✅ dim_products creada: {len(dim_products)} registros")
    print(f"Categorías únicas: {dim_products['category'].nunique()}")
else:
    print("❌ No se encontraron products_df o suppliers_df")

Creando dim_products...
✅ dim_products creada: 45 registros
Categorías únicas: 16


### Tabla dim_customers

In [6]:
# Celda 5: Creación de dim_customers - Clientes con geografía jerárquica
print("Creando dim_customers...")

if customers_df is not None:
    dim_customers = pd.DataFrame({
        'sk_customer': range(1, len(customers_df) + 1),
        'customer_id': customers_df['id'],
        'company_name': customers_df['company'],
        'full_name': customers_df['first_name'] + ' ' + customers_df['last_name'],
        'email_address': customers_df['email_address'],
        'job_title': customers_df['job_title'],
        'city': customers_df['city'],
        'state_province': customers_df['state_province'],
        'country_region': customers_df['country_region'],
        'zip_postal_code': customers_df['zip_postal_code'],
        'geography': customers_df['city'] + ', ' + customers_df['state_province'] + ', ' + customers_df['country_region']
    })
    
    # Guardar SOLO en el diccionario principal
    dataframes['dim_customers'] = dim_customers
    
    print(f"✅ dim_customers creada: {len(dim_customers)} registros")
    print(f"Países: {dim_customers['country_region'].unique()}")
else:
    print("❌ No se encontró customers_df")

Creando dim_customers...
✅ dim_customers creada: 29 registros
Países: <StringArray>
['USA']
Length: 1, dtype: str


### Tabla dim_employees

In [7]:
# Celda 6: Creación de dim_employees - Empleados con privilegios
print("Creando dim_employees...")

if employees_df is not None:
    # Merge con privilegios si existen
    if employee_privileges_df is not None and privileges_df is not None:
        employees_with_priv = employees_df.merge(
            employee_privileges_df,
            left_on='id',
            right_on='employee_id',
            how='left'
        ).merge(
            privileges_df[['id', 'privilege_name']],
            left_on='privilege_id',
            right_on='id',
            how='left',
            suffixes=('', '_priv')
        )
        
        # Agrupar privilegios por empleado (pueden tener múltiples)
        employee_privileges_grouped = employees_with_priv.groupby('id').agg({
            'privilege_name': lambda x: ', '.join(x.dropna()) if len(x.dropna()) > 0 else None
        }).reset_index()
        
        employees_final = employees_df.merge(
            employee_privileges_grouped,
            on='id',
            how='left'
        )
    else:
        employees_final = employees_df.copy()
        employees_final['privilege_name'] = None
    
    dim_employees = pd.DataFrame({
        'sk_employee': range(1, len(employees_df) + 1),
        'employee_id': employees_final['id'],
        'full_name': employees_final['first_name'] + ' ' + employees_final['last_name'],
        'job_title': employees_final['job_title'],
        'privilege': employees_final.get('privilege_name', None),
        'email_address': employees_final['email_address'],
        'city': employees_final['city']
    })
    
    # Guardar SOLO en el diccionario principal
    dataframes['dim_employees'] = dim_employees
    
    print(f"✅ dim_employees creada: {len(dim_employees)} registros")
    print(f"Empleados con privilegios: {dim_employees['privilege'].notna().sum()}")
else:
    print("❌ No se encontró employees_df")

Creando dim_employees...
✅ dim_employees creada: 9 registros
Empleados con privilegios: 1


### Tabla dim_suppliers y dim_shippers

In [8]:
# Celda 7: Creación de dim_suppliers y dim_shippers
print("Creando dimensiones de proveedores y transportistas...")

# dim_suppliers - Proveedores
if suppliers_df is not None:
    dim_suppliers = pd.DataFrame({
        'sk_supplier': range(1, len(suppliers_df) + 1),
        'supplier_id': suppliers_df['id'],
        'company': suppliers_df['company'],
        'first_name': suppliers_df['first_name'],
        'last_name': suppliers_df['last_name'],
        'job_title': suppliers_df['job_title']
    })
    # Guardar SOLO en el diccionario principal
    dataframes['dim_suppliers'] = dim_suppliers
    print(f"✅ dim_suppliers creada: {len(dim_suppliers)} registros")
else:
    print("❌ No se encontró suppliers_df")

# dim_shippers - Transportistas
if shippers_df is not None:
    dim_shippers = pd.DataFrame({
        'sk_shipper': range(1, len(shippers_df) + 1),
        'shipper_id': shippers_df['id'],
        'company': shippers_df['company']
    })
    # Guardar SOLO en el diccionario principal
    dataframes['dim_shippers'] = dim_shippers
    print(f"✅ dim_shippers creada: {len(dim_shippers)} registros")
else:
    print("❌ No se encontró shippers_df")

Creando dimensiones de proveedores y transportistas...
✅ dim_suppliers creada: 10 registros
✅ dim_shippers creada: 3 registros


### Mapeo para las claves subrogadas de cada tabla

In [9]:
# Celda 8: Preparación de mapas de lookup para claves subrogadas
print("Preparando mapas de lookup para claves subrogadas...")

# Verificar que todas las dimensiones necesarias existen en dataframes
dimensiones_requeridas = ['dim_customers', 'dim_employees', 'dim_products', 'dim_shippers', 'dim_date', 'dim_suppliers']
mapas = {}

for dim in dimensiones_requeridas:
    if dim in dataframes:
        df_dim = dataframes[dim]
        
        if dim == 'dim_date':
            # Para fechas, el lookup es por date_actual
            id_col = 'date_actual'
            sk_col = 'sk_date'
        else:
            # Para las demás dimensiones, identificar la columna de ID original y SK
            # La columna SK siempre empieza con 'sk_'
            sk_col = [col for col in df_dim.columns if col.startswith('sk_')][0]
            
            # La columna de ID original termina con '_id' y no es la SK
            id_col_candidates = [col for col in df_dim.columns if col.endswith('_id') and col != sk_col]
            
            if id_col_candidates:
                id_col = id_col_candidates[0]
            else:
                # Si no encuentra columna con '_id', busca otras alternativas
                if 'customer_id' in df_dim.columns:
                    id_col = 'customer_id'
                elif 'employee_id' in df_dim.columns:
                    id_col = 'employee_id'
                elif 'product_id' in df_dim.columns:
                    id_col = 'product_id'
                elif 'shipper_id' in df_dim.columns:
                    id_col = 'shipper_id'
                elif 'supplier_id' in df_dim.columns:
                    id_col = 'supplier_id'
                else:
                    print(f"⚠️ No se pudo identificar columna ID para {dim}")
                    continue
        
        # Crear el mapa de lookup
        mapas[dim] = dict(zip(df_dim[id_col], df_dim[sk_col]))
        print(f"✅ Mapa creado para {dim}: {len(mapas[dim])} entradas (ID: {id_col} -> SK: {sk_col})")
    else:
        print(f"⚠️ {dim} no encontrada en dataframes, no se creará mapa")

# Guardar los mapas en dataframes (son útiles para las tablas de hechos)
dataframes['lookup_maps'] = mapas

# Verificar que todos los mapas necesarios están presentes
print("\n📊 Mapas disponibles:")
for mapa in mapas:
    ejemplo_key = list(mapas[mapa].keys())[0] if mapas[mapa] else "vacío"
    ejemplo_val = mapas[mapa][ejemplo_key] if mapas[mapa] else "vacío"
    print(f"  • {mapa}: {len(mapas[mapa])} entradas (ej: {ejemplo_key} -> {ejemplo_val})")

print("\n✅ Mapas de lookup preparados")

Preparando mapas de lookup para claves subrogadas...
✅ Mapa creado para dim_customers: 29 entradas (ID: customer_id -> SK: sk_customer)
✅ Mapa creado para dim_employees: 9 entradas (ID: employee_id -> SK: sk_employee)
✅ Mapa creado para dim_products: 45 entradas (ID: product_id -> SK: sk_product)
✅ Mapa creado para dim_shippers: 3 entradas (ID: shipper_id -> SK: sk_shipper)
✅ Mapa creado para dim_date: 137 entradas (ID: date_actual -> SK: sk_date)
✅ Mapa creado para dim_suppliers: 10 entradas (ID: supplier_id -> SK: sk_supplier)

📊 Mapas disponibles:
  • dim_customers: 29 entradas (ej: 1 -> 1)
  • dim_employees: 9 entradas (ej: 1 -> 1)
  • dim_products: 45 entradas (ej: 1 -> 1)
  • dim_shippers: 3 entradas (ej: 1 -> 1)
  • dim_date: 137 entradas (ej: 2006-01-15 00:00:00 -> 20060115)
  • dim_suppliers: 10 entradas (ej: 1 -> 1)

✅ Mapas de lookup preparados


### Tabla fact_sales

In [10]:
# Celda 9: Creación de fact_sales - Ventas Detalladas
print("Creando fact_sales...")

# Obtener mapas de lookup
mapas = dataframes.get('lookup_maps', {})

if all([order_details_df is not None, orders_df is not None, mapas]):
    # Merge orders con order_details
    sales_base = order_details_df.merge(
        orders_df,
        left_on='order_id',
        right_on='id',
        how='inner',
        suffixes=('_detail', '_order')
    )
    
    # Convertir fechas
    sales_base['order_date_dt'] = pd.to_datetime(sales_base['order_date'], errors='coerce')
    
    # Asignar claves subrogadas usando los mapas
    fact_sales = pd.DataFrame()
    fact_sales['sk_sale'] = range(1, len(sales_base) + 1)
    fact_sales['order_id'] = sales_base['order_id']  # Clave degenerada
    fact_sales['sk_date'] = sales_base['order_date_dt'].map(mapas.get('dim_date', {}))
    fact_sales['sk_customer'] = sales_base['customer_id'].map(mapas.get('dim_customers', {}))
    fact_sales['sk_employee'] = sales_base['employee_id'].map(mapas.get('dim_employees', {}))
    fact_sales['sk_product'] = sales_base['product_id'].map(mapas.get('dim_products', {}))
    fact_sales['sk_shipper'] = sales_base['shipper_id'].map(mapas.get('dim_shippers', {}))
    fact_sales['quantity'] = sales_base['quantity']
    fact_sales['unit_price'] = sales_base['unit_price']
    fact_sales['discount'] = sales_base['discount']
    fact_sales['line_total'] = sales_base['line_total']
    
    # Prorratear shipping_fee por línea (basado en proporción del total)
    # Calcular total por orden
    order_totals = fact_sales.groupby('order_id')['line_total'].sum().reset_index()
    order_totals.columns = ['order_id', 'order_total']
    
    # Merge con shipping_fee de orders
    order_shipping = orders_df[['id', 'shipping_fee']].copy()
    order_shipping.columns = ['order_id', 'shipping_fee']
    
    fact_sales = fact_sales.merge(order_totals, on='order_id', how='left')
    fact_sales = fact_sales.merge(order_shipping, on='order_id', how='left')
    
    # Prorratear: (line_total / order_total) * shipping_fee
    fact_sales['shipping_fee'] = np.where(
        fact_sales['order_total'] > 0,
        (fact_sales['line_total'] / fact_sales['order_total']) * fact_sales['shipping_fee'],
        0
    )
    
    # Eliminar columna temporal
    fact_sales = fact_sales.drop(columns=['order_total'])
    
    # Guardar SOLO en el diccionario principal
    dataframes['fact_sales'] = fact_sales
    
    print(f"✅ fact_sales creada: {len(fact_sales)} registros")
    print(f"Rango de ventas: ${fact_sales['line_total'].min():.2f} a ${fact_sales['line_total'].max():.2f}")
    print(f"Total ventas: ${fact_sales['line_total'].sum():.2f}")
else:
    print("❌ Faltan dataframes necesarios para fact_sales")

Creando fact_sales...
✅ fact_sales creada: 58 registros
Rango de ventas: $0.00 a $13800.00
Total ventas: $68137.00


### Tabla fact_inventory_movements

In [11]:
# Celda 10: Creación de fact_inventory_movements - Movimientos de Stock
print("Creando fact_inventory_movements...")

mapas = dataframes.get('lookup_maps', {})

if all([inventory_transactions_df is not None, inventory_transaction_types_df is not None, mapas]):
    # Merge con tipo de transacción
    inv_with_type = inventory_transactions_df.merge(
        inventory_transaction_types_df,
        left_on='transaction_type',
        right_on='id',
        how='left'
    )
    
    # Convertir fechas
    inv_with_type['date_dt'] = pd.to_datetime(inv_with_type['transaction_created_date'], errors='coerce')
    
    fact_inventory = pd.DataFrame()
    fact_inventory['sk_inventory'] = range(1, len(inv_with_type) + 1)
    fact_inventory['sk_date'] = inv_with_type['date_dt'].map(mapas.get('dim_date', {}))
    fact_inventory['sk_product'] = inv_with_type['product_id'].map(mapas.get('dim_products', {}))
    fact_inventory['transaction_type'] = inv_with_type['type_name']
    fact_inventory['quantity'] = inv_with_type['quantity']
    fact_inventory['is_backorder'] = inv_with_type['is_backorder']
    
    # Guardar SOLO en el diccionario principal
    dataframes['fact_inventory_movements'] = fact_inventory
    
    print(f"✅ fact_inventory_movements creada: {len(fact_inventory)} registros")
    print(f"Tipos de transacción: {fact_inventory['transaction_type'].unique()}")
    print(f"Total unidades movidas: {fact_inventory['quantity'].sum()}")
else:
    print("❌ Faltan dataframes necesarios para fact_inventory_movements")

Creando fact_inventory_movements...
✅ fact_inventory_movements creada: 102 registros
Tipos de transacción: <StringArray>
['Purchased', 'Sold', 'On Hold']
Length: 3, dtype: str
Total unidades movidas: 6615


### Tabla fact_purchases

In [12]:
# Celda 11: Creación de fact_purchases - Compras a Proveedores
print("Creando fact_purchases...")

mapas = dataframes.get('lookup_maps', {})

if all([purchase_order_details_df is not None, purchase_orders_df is not None, mapas]):
    # Merge purchase orders con details
    purchases_base = purchase_order_details_df.merge(
        purchase_orders_df,
        left_on='purchase_order_id',
        right_on='id',
        how='inner',
        suffixes=('_detail', '_po')
    )
    
    # Convertir fechas
    purchases_base['date_dt'] = pd.to_datetime(purchases_base['creation_date'], errors='coerce')
    
    fact_purchases = pd.DataFrame()
    fact_purchases['sk_purchase'] = range(1, len(purchases_base) + 1)
    fact_purchases['sk_date'] = purchases_base['date_dt'].map(mapas.get('dim_date', {}))
    fact_purchases['sk_product'] = purchases_base['product_id'].map(mapas.get('dim_products', {}))
    fact_purchases['sk_supplier'] = purchases_base['supplier_id'].map(mapas.get('dim_suppliers', {}))
    fact_purchases['unit_cost'] = purchases_base['unit_cost']
    fact_purchases['quantity'] = purchases_base['quantity']
    fact_purchases['purchase_order_id'] = purchases_base['purchase_order_id']
    fact_purchases['date_received'] = purchases_base['date_received']
    fact_purchases['posted_to_inventory'] = purchases_base['posted_to_inventory']
    
    # Guardar SOLO en el diccionario principal
    dataframes['fact_purchases'] = fact_purchases
    
    print(f"✅ fact_purchases creada: {len(fact_purchases)} registros")
    print(f"Costo total: ${(fact_purchases['unit_cost'] * fact_purchases['quantity']).sum():.2f}")
else:
    print("❌ Faltan dataframes necesarios para fact_purchases")

Creando fact_purchases...
✅ fact_purchases creada: 55 registros
Costo total: $77921.00


## Validación de claves foraneas 

In [13]:
# Celda 12: Validación de integridad de claves foráneas
print("Validando integridad de claves foráneas...")
print("=" * 60)

# Mostrar qué dimensiones están disponibles en dataframes
print("Dimensiones disponibles en dataframes:")
dimensiones_disponibles = [k for k in dataframes.keys() if k.startswith('dim_')]
for dim in sorted(dimensiones_disponibles):
    print(f"  • {dim}")

# Mostrar qué tablas de hechos están disponibles en dataframes
print("\nTablas de hechos disponibles en dataframes:")
hechos_disponibles = [k for k in dataframes.keys() if k.startswith('fact_')]
for hecho in sorted(hechos_disponibles):
    print(f"  • {hecho}")

print("\n" + "=" * 60)

# Crear un diccionario de mapeo de nombres de columnas a dimensiones
mapeo_columnas_a_dimensiones = {
    'sk_date': 'dim_date',
    'sk_customer': 'dim_customers',
    'sk_employee': 'dim_employees',
    'sk_product': 'dim_products',
    'sk_shipper': 'dim_shippers',
    'sk_supplier': 'dim_suppliers',
    'sk_sale': None,
    'sk_inventory': None,
    'sk_purchase': None
}

tablas_hechos = ['fact_sales', 'fact_inventory_movements', 'fact_purchases']

for hecho in tablas_hechos:
    if hecho in dataframes:
        df_hecho = dataframes[hecho]
        print(f"\n📊 Validando {hecho} ({len(df_hecho)} registros)")
        print("-" * 40)
        
        fk_cols = [col for col in df_hecho.columns if col.startswith('sk_')
                  and col not in ['sk_sale', 'sk_inventory', 'sk_purchase']]
        
        for fk in fk_cols:
            dim_name = mapeo_columnas_a_dimensiones.get(fk)
            
            if dim_name and dim_name in dataframes:
                df_dim = dataframes[dim_name]
                valores_validos = set(df_dim[fk].unique())
                valores_en_hecho = set(df_hecho[fk].dropna().unique())
                
                valores_invalidos = valores_en_hecho - valores_validos
                if valores_invalidos:
                    print(f"❌ {fk}: {len(valores_invalidos)} valores inválidos")
                    if len(valores_invalidos) <= 5:
                        print(f"   Inválidos: {sorted(valores_invalidos)}")
                else:
                    print(f"✅ {fk}: todos los valores son válidos")
                
                nulos = df_hecho[fk].isna().sum()
                if nulos > 0:
                    pct_nulos = (nulos / len(df_hecho)) * 100
                    print(f"⚠️ {fk}: {nulos} valores nulos ({pct_nulos:.1f}%)")
            elif dim_name:
                print(f"❌ {fk}: No se encontró la dimensión '{dim_name}' en dataframes")
            else:
                print(f"ℹ️ {fk}: No requiere validación (clave primaria)")

# Validaciones adicionales
print("\n" + "=" * 60)
print("VERIFICACIÓN DE CONSISTENCIA ENTRE TABLAS DE HECHOS")
print("=" * 60)

if 'fact_sales' in dataframes and 'dim_products' in dataframes:
    productos_en_ventas = set(dataframes['fact_sales']['sk_product'].dropna().unique())
    productos_en_dim = set(dataframes['dim_products']['sk_product'].unique())
    productos_faltantes = productos_en_ventas - productos_en_dim
    if productos_faltantes:
        print(f"❌ fact_sales tiene {len(productos_faltantes)} productos que no están en dim_products")
    else:
        print(f"✅ Todos los productos en fact_sales existen en dim_products")

if 'fact_sales' in dataframes and 'dim_customers' in dataframes:
    clientes_en_ventas = set(dataframes['fact_sales']['sk_customer'].dropna().unique())
    clientes_en_dim = set(dataframes['dim_customers']['sk_customer'].unique())
    clientes_faltantes = clientes_en_ventas - clientes_en_dim
    if clientes_faltantes:
        print(f"❌ fact_sales tiene {len(clientes_faltantes)} clientes que no están en dim_customers")
    else:
        print(f"✅ Todos los clientes en fact_sales existen en dim_customers")

if 'fact_sales' in dataframes and 'dim_employees' in dataframes:
    empleados_en_ventas = set(dataframes['fact_sales']['sk_employee'].dropna().unique())
    empleados_en_dim = set(dataframes['dim_employees']['sk_employee'].unique())
    empleados_faltantes = empleados_en_ventas - empleados_en_dim
    if empleados_faltantes:
        print(f"❌ fact_sales tiene {len(empleados_faltantes)} empleados que no están en dim_employees")
    else:
        print(f"✅ Todos los empleados en fact_sales existen en dim_employees")

Validando integridad de claves foráneas...
Dimensiones disponibles en dataframes:
  • dim_customers
  • dim_date
  • dim_employees
  • dim_products
  • dim_shippers
  • dim_suppliers

Tablas de hechos disponibles en dataframes:
  • fact_inventory_movements
  • fact_purchases
  • fact_sales


📊 Validando fact_sales (58 registros)
----------------------------------------
✅ sk_date: todos los valores son válidos
✅ sk_customer: todos los valores son válidos
✅ sk_employee: todos los valores son válidos
✅ sk_product: todos los valores son válidos
✅ sk_shipper: todos los valores son válidos
⚠️ sk_shipper: 7 valores nulos (12.1%)

📊 Validando fact_inventory_movements (102 registros)
----------------------------------------
✅ sk_date: todos los valores son válidos
✅ sk_product: todos los valores son válidos

📊 Validando fact_purchases (55 registros)
----------------------------------------
✅ sk_date: todos los valores son válidos
✅ sk_product: todos los valores son válidos
✅ sk_supplier: todos 

## Resumen de total de tablas creadas

In [14]:
# Celda 13: Resumen final de tablas creadas
print("=" * 60)
print("RESUMEN FINAL - TABLAS OLAP CREADAS")
print("=" * 60)

# Dimensiones
print("\n📊 DIMENSIONES:")
dimensiones_creadas = [k for k in dataframes.keys() if k.startswith('dim_')]
for dim in sorted(dimensiones_creadas):
    df = dataframes[dim]
    print(f"  • {dim}: {len(df):>5} registros | Columnas: {list(df.columns)}")

# Tablas de hechos
print("\n📈 TABLAS DE HECHOS:")
hechos_creados = [k for k in dataframes.keys() if k.startswith('fact_')]
for hecho in sorted(hechos_creados):
    df = dataframes[hecho]
    print(f"  • {hecho}: {len(df):>5} registros | Columnas: {list(df.columns)}")

# Mostrar solo las claves de tablas dimensionales y de hechos
print(f"\n📌 Total de tablas OLAP en el diccionario: {len(dimensiones_creadas) + len(hechos_creados)}")
print(f"Claves de tablas OLAP: {sorted([k for k in dataframes.keys() if k.startswith(('dim_', 'fact_'))])}")

RESUMEN FINAL - TABLAS OLAP CREADAS

📊 DIMENSIONES:
  • dim_customers:    29 registros | Columnas: ['sk_customer', 'customer_id', 'company_name', 'full_name', 'email_address', 'job_title', 'city', 'state_province', 'country_region', 'zip_postal_code', 'geography']
  • dim_date:   137 registros | Columnas: ['sk_date', 'date_actual', 'year', 'quarter', 'month', 'month_name', 'day', 'day_name', 'week', 'is_weekend']
  • dim_employees:     9 registros | Columnas: ['sk_employee', 'employee_id', 'full_name', 'job_title', 'privilege', 'email_address', 'city']
  • dim_products:    45 registros | Columnas: ['sk_product', 'product_id', 'product_name', 'category', 'supplier_company', 'list_price', 'standard_cost', 'quantity_per_unit', 'discontinued']
  • dim_shippers:     3 registros | Columnas: ['sk_shipper', 'shipper_id', 'company']
  • dim_suppliers:    10 registros | Columnas: ['sk_supplier', 'supplier_id', 'company', 'first_name', 'last_name', 'job_title']

📈 TABLAS DE HECHOS:
  • fact_inven

## Cargar datos en data/03_gold

In [15]:
# Identificar tablas OLAP
tablas_olap = [k for k in dataframes.keys() if k.startswith(('dim_', 'fact_'))]

print(f"\nTablas a guardar ({len(tablas_olap)}): {tablas_olap}\n")

# Guardar cada tabla
for tabla in tablas_olap:
    df = dataframes[tabla]
    output_file = get_gold_file(tabla)
    df.to_csv(output_file, index=False, encoding='utf-8')
    print(f"  ✅ {tabla}: {len(df)} registros, {len(df.columns)} columnas -> {output_file.name}")

print(f"\n✅ Todas las tablas guardadas en: {GOLD_PATH}")


Tablas a guardar (9): ['dim_date', 'dim_products', 'dim_customers', 'dim_employees', 'dim_suppliers', 'dim_shippers', 'fact_sales', 'fact_inventory_movements', 'fact_purchases']

  ✅ dim_date: 137 registros, 10 columnas -> dim_date.csv
  ✅ dim_products: 45 registros, 9 columnas -> dim_products.csv
  ✅ dim_customers: 29 registros, 11 columnas -> dim_customers.csv
  ✅ dim_employees: 9 registros, 7 columnas -> dim_employees.csv
  ✅ dim_suppliers: 10 registros, 6 columnas -> dim_suppliers.csv
  ✅ dim_shippers: 3 registros, 3 columnas -> dim_shippers.csv
  ✅ fact_sales: 58 registros, 12 columnas -> fact_sales.csv
  ✅ fact_inventory_movements: 102 registros, 6 columnas -> fact_inventory_movements.csv
  ✅ fact_purchases: 55 registros, 9 columnas -> fact_purchases.csv

✅ Todas las tablas guardadas en: D:\proyectos de github\proyecto northwind\data\03_gold
